# Redressement BACI des flux de commerce international

Reconstruction des flux bilatéraux réconciliés (méthodologie CEPII *BACI*) à partir des déclarations brutes COMTRADE stockées dans un catalogue DuckLake et des variables de gravité CEPII.

Le pipeline enchaîne : conversion des quantités en tonnes → estimation des taux CAF par équation de gravité → fobisation des importations → évaluation de la qualité de déclaration (ANOVA) → réconciliation pondérée des flux miroirs → réallocation des zones non spécifiées.

**Prérequis :** le catalogue COMTRADE (`data/comtrade.ducklake`) doit avoir été peuplé au préalable via `scripts/download_comtrade.py`.

In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

# Racine du dépôt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# I/O DuckLake et configuration : côté script ; méthodologie : côté module
from scripts.process_baci import (
    load_baci_config,
    baci_config_from_params,
    _read_comtrade_fact_table,
)
from macroforecast.storage2 import Loader, write_dataframe
from macroforecast.trade.processing import (
    run_baci,
    load_gravity_data,
    required_columns,
    DEFAULT_CONFIG,
)

In [ ]:
# Chargement de la configuration (chemins + paramètres méthodologiques)
config = load_baci_config(ROOT / "config" / "baci.yaml")
paths = config["paths"]
baci_config = baci_config_from_params(config.get("parameters"))


def resolve(p):
    p = Path(p)
    return p if p.is_absolute() else ROOT / p


print("BUCKET :", config["BUCKET"])
print("Schéma source COMTRADE :", paths["comtrade_schema"])
print("Distance utilisée :", baci_config.distance_column)

In [ ]:
# Lecture des fichiers Excel CEPII (loader bucket-aware) puis assemblage de la gravité
excel_loader = Loader()
dist = excel_loader.load(str(resolve(paths["dist_cepii"])), bucket=config["BUCKET"])
geo = excel_loader.load(str(resolve(paths["geo_cepii"])), bucket=config["BUCKET"])

gravity = load_gravity_data(dist, geo, baci_config)
print("Table de gravité :", gravity.shape)
gravity.head()

In [ ]:
# Inspection de la source COMTRADE (nombre de déclarations)


def _attach(catalog, data, alias, *, read_only):
    """Attache un catalogue DuckLake fichier à une nouvelle connexion.

    OVERRIDE_DATA_PATH tolère un chemin de données normalisé différemment de
    celui stocké dans le catalogue (indispensable sous Windows/OneDrive).
    """
    conn = duckdb.connect()
    conn.execute("INSTALL ducklake; LOAD ducklake;")
    options = f"DATA_PATH '{Path(data).as_posix()}/', OVERRIDE_DATA_PATH true"
    if read_only:
        options += ", READ_ONLY"
    conn.execute(f"ATTACH 'ducklake:{Path(catalog).as_posix()}' AS {alias} ({options})")
    return conn


def attach_readonly(catalog, data, alias):
    """Attache un catalogue DuckLake en lecture seule."""
    return _attach(catalog, data, alias, read_only=True)


def attach_writable(catalog, data, alias, schema):
    """Attache un catalogue DuckLake en écriture et garantit le schéma cible."""
    Path(data).mkdir(parents=True, exist_ok=True)
    conn = _attach(catalog, data, alias, read_only=False)
    conn.execute(f"CREATE SCHEMA IF NOT EXISTS {alias}.{schema}")
    return conn


conn = attach_readonly(resolve(paths["comtrade_catalog"]), resolve(paths["comtrade_data"]), "src")
n = conn.execute(f"SELECT count(*) FROM src.{paths['comtrade_schema']}.fact_table").fetchone()[0]
conn.close()
print(f"{n:,} déclarations COMTRADE")

In [ ]:
# Lecture de la table de faits COMTRADE puis redressement BACI de bout en bout.
# Les connexions appartiennent à l'appelant : write_dataframe ne les gère pas.
src = attach_readonly(resolve(paths["comtrade_catalog"]), resolve(paths["comtrade_data"]), "src")
res = attach_writable(
    resolve(paths["result_catalog"]),
    resolve(paths["result_data"]),
    "res",
    paths["result_schema"],
)
try:
    comtrade = _read_comtrade_fact_table(
        conn=src,
        source_schema=paths["comtrade_schema"],
        columns=required_columns(baci_config),
    )
    reconciled, report = run_baci(comtrade, dist, geo, config=baci_config)

    # Écriture du résultat dans le catalogue DuckLake (création puis upsert)
    report.created = write_dataframe(
        res,
        reconciled,
        baci_config.primary_keys,
        catalog_alias="res",
        schema=paths["result_schema"],
    )
finally:
    src.close()
    res.close()
report

In [ ]:
# Relecture du résultat et contrôles de cohérence
rconn = attach_readonly(resolve(paths["result_catalog"]), resolve(paths["result_data"]), "res")
baci = rconn.execute(f"SELECT * FROM res.{paths['result_schema']}.fact_table").df()
rconn.close()

print("Flux réconciliés :", baci.shape)
print("Taux de fret moyen estimé : {:.2%}".format(report.mean_freight_rate))
keys = ["exporter", "importer", "product", "year"]
print("Clés (i, j, k, t) uniques :", not baci.duplicated(subset=keys).any())
print("Valeurs réconciliées négatives :", int((baci["reconciled_value"] < 0).sum()))
baci.head()

Contrôles attendus : un **taux de fret moyen de l'ordre de 3 %** (cohérent avec la note méthodologique), des **clés `(exportateur, importateur, produit, année)` uniques** et **aucune valeur réconciliée négative**. Ré-exécuter la cellule `run_baci` réalise un *upsert* (`created=False`) plutôt qu'une recréation du schéma.